# Spots Strategy Simulation
Compare Random, Greedy, and Conservative strategies over N games.
All strategies play in the same game; results show seat-adjusted win rates.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))
# Ensure strat_hacking root is on path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from spots.sim import simulate
from spots.strategies import RandomStrategy, GreedyStrategy, ConservativeStrategy

In [ ]:
# ── Run simulation ─────────────────────────────────────────────────────────────
N_GAMES = 10_000

factories = [
    lambda rng: RandomStrategy(rng),
    lambda rng: GreedyStrategy(rng),
    lambda rng: ConservativeStrategy(rng),
    lambda rng: GreedyStrategy(rng),   # second Greedy as baseline
]
names = ['Random', 'Greedy A', 'Conservative', 'Greedy B']

print(f'Running {N_GAMES:,} games…')
stats = simulate(factories, names, n_games=N_GAMES, seed=42)
print(stats)

In [ ]:
# ── Bar chart: win rates ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

colours = ['#7f7f7f', '#2ca02c', '#d62728', '#1f77b4']
x = np.arange(len(names))

# Win rate
ax = axes[0]
bars = ax.bar(x, [r * 100 for r in stats.win_rates], color=colours)
ax.axhline(100 / len(names), color='black', linestyle='--', linewidth=0.8, label='Equal split')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15)
ax.set_ylabel('Win rate (%)')
ax.set_title('Win Rate')
for bar, v in zip(bars, stats.win_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{v:.1%}', ha='center', va='bottom', fontsize=10)

# Average dogs scored
ax = axes[1]
bars = ax.bar(x, stats.avg_score, color=colours)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15)
ax.set_ylabel('Avg dogs scored')
ax.set_title('Average Score')
for bar, v in zip(bars, stats.avg_score):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{v:.2f}', ha='center', va='bottom', fontsize=10)

# Average busts
ax = axes[2]
bars = ax.bar(x, stats.avg_busts, color=colours)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15)
ax.set_ylabel('Avg busts per game')
ax.set_title('Bust Rate')
for bar, v in zip(bars, stats.avg_busts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{v:.2f}', ha='center', va='bottom', fontsize=10)

fig.suptitle(f'Spots Strategy Comparison — {N_GAMES:,} games', fontsize=13, y=1.02)
fig.tight_layout()
out_path = os.path.join(ROOT, 'spots', 'strategy_comparison.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

In [ ]:
# ── Treats vs wins scatter (per game) ─────────────────────────────────────────
# Re-run a smaller batch collecting per-game detail
import random as _random
from spots.sim import run_game

N_DETAIL = 2_000
rng = _random.Random(99)
results = []
for _ in range(N_DETAIL):
    g_rng = _random.Random(rng.randint(0, 2**31))
    strats = [RandomStrategy(g_rng), GreedyStrategy(g_rng),
               ConservativeStrategy(g_rng), GreedyStrategy(g_rng)]
    r = run_game(strats, g_rng)
    results.append(r)

fig, ax = plt.subplots(figsize=(7, 5))
for i, (name, colour) in enumerate(zip(names, colours)):
    sc = [r.scores[i] for r in results]
    tr = [r.treats[i] for r in results]
    ax.scatter(tr, sc, alpha=0.1, s=8, color=colour, label=name)
ax.set_xlabel('Treats at game end')
ax.set_ylabel('Dogs scored')
ax.set_title(f'Score vs Treats ({N_DETAIL:,} games)')
ax.legend(markerscale=4)
fig.tight_layout()
plt.show()